In [1]:
import random
import time
import re
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "distilbert-base-uncased"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
subset_size = 256
max_length = 64
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 64 if device == "mps" else 32
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "subset_size": subset_size,
    "max_length": max_length,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})


{'model_name': 'distilbert-base-uncased', 'dataset': 'glue/stsb', 'split': 'validation', 'subset_size': 256, 'max_length': 64, 'device': 'mps', 'batch_size': 64, 'seed': 42}


In [2]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()

def minimal_clean(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def word_count(text):
    return len(re.findall(r"\b\w+\b", str(text)))

df["sentence1"] = df["sentence1"].map(minimal_clean)
df["sentence2"] = df["sentence2"].map(minimal_clean)
df = df[(df["sentence1"] != "") & (df["sentence2"] != "")].reset_index(drop=True)

subset_df = df.head(subset_size).copy().reset_index(drop=True)
subset_df["len1_words"] = subset_df["sentence1"].map(word_count)
subset_df["len2_words"] = subset_df["sentence2"].map(word_count)
subset_df["avg_words"] = (subset_df["len1_words"] + subset_df["len2_words"]) / 2.0
subset_df["len_gap_words"] = (subset_df["len1_words"] - subset_df["len2_words"]).abs()

print({
    "original_num_examples": len(df),
    "subset_num_examples": len(subset_df),
    "columns": subset_df.columns.tolist(),
})
print(subset_df[["sentence1", "sentence2", "label", "len1_words", "len2_words", "avg_words", "len_gap_words"]].head(10))


{'original_num_examples': 1500, 'subset_num_examples': 256, 'columns': ['sentence1', 'sentence2', 'label', 'len1_words', 'len2_words', 'avg_words', 'len_gap_words']}
                              sentence1  \
0     A man with a hard hat is dancing.   
1      A young child is riding a horse.   
2  A man is feeding a mouse to a snake.   
3        A woman is playing the guitar.   
4         A woman is playing the flute.   
5          A woman is cutting an onion.   
6       A man is erasing a chalk board.   
7            A woman is carrying a boy.   
8        Three men are playing guitars.   
9               A woman peels a potato.   

                                  sentence2  label  len1_words  len2_words  \
0      A man wearing a hard hat is dancing.  5.000           8           8   
1                A child is riding a horse.  4.750           7           6   
2  The man is feeding a mouse to the snake.  5.000           9           9   
3                  A man is playing guitar.  2.4

In [3]:
length_stats = {
    "len1_words_mean": float(subset_df["len1_words"].mean()),
    "len1_words_median": float(subset_df["len1_words"].median()),
    "len1_words_min": int(subset_df["len1_words"].min()),
    "len1_words_max": int(subset_df["len1_words"].max()),
    "len2_words_mean": float(subset_df["len2_words"].mean()),
    "len2_words_median": float(subset_df["len2_words"].median()),
    "len2_words_min": int(subset_df["len2_words"].min()),
    "len2_words_max": int(subset_df["len2_words"].max()),
    "avg_words_mean": float(subset_df["avg_words"].mean()),
    "avg_words_median": float(subset_df["avg_words"].median()),
    "len_gap_words_mean": float(subset_df["len_gap_words"].mean()),
    "len_gap_words_median": float(subset_df["len_gap_words"].median()),
}
print(length_stats)


{'len1_words_mean': 6.80078125, 'len1_words_median': 6.0, 'len1_words_min': 3, 'len1_words_max': 15, 'len2_words_mean': 6.7109375, 'len2_words_median': 6.0, 'len2_words_min': 3, 'len2_words_max': 17, 'avg_words_mean': 6.755859375, 'avg_words_median': 6.5, 'len_gap_words_mean': 1.33984375, 'len_gap_words_median': 1.0}


In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()
print(model_name)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


distilbert-base-uncased


In [5]:
sentences1 = subset_df["sentence1"].tolist()
sentences2 = subset_df["sentence2"].tolist()
labels = subset_df["label"].to_numpy(dtype=np.float32)

def encode_cls(sentences, tokenizer, model, device, batch_size=32, max_length=64):
    all_embeddings = []
    with torch.no_grad():
        for i in range(0, len(sentences), batch_size):
            batch_sentences = sentences[i:i + batch_size]
            encoded = tokenizer(
                batch_sentences,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            encoded = {k: v.to(device) for k, v in encoded.items()}
            outputs = model(**encoded)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls_embeddings.detach().cpu())
    return torch.cat(all_embeddings, dim=0).numpy()

emb1 = encode_cls(sentences1, tokenizer, model, device, batch_size=batch_size, max_length=max_length)
emb2 = encode_cls(sentences2, tokenizer, model, device, batch_size=batch_size, max_length=max_length)

emb1_norm = emb1 / np.clip(np.linalg.norm(emb1, axis=1, keepdims=True), 1e-12, None)
emb2_norm = emb2 / np.clip(np.linalg.norm(emb2, axis=1, keepdims=True), 1e-12, None)

cosine_similarity = np.sum(emb1_norm * emb2_norm, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)


In [6]:
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic
mae = float(np.mean(np.abs(predicted_score_0_5 - labels)))

emb1_raw_norms = np.linalg.norm(emb1, axis=1)
emb2_raw_norms = np.linalg.norm(emb2, axis=1)
embedding_norm_stats = {
    "emb1_norm_mean": float(emb1_raw_norms.mean()),
    "emb1_norm_std": float(emb1_raw_norms.std()),
    "emb1_norm_min": float(emb1_raw_norms.min()),
    "emb1_norm_max": float(emb1_raw_norms.max()),
    "emb2_norm_mean": float(emb2_raw_norms.mean()),
    "emb2_norm_std": float(emb2_raw_norms.std()),
    "emb2_norm_min": float(emb2_raw_norms.min()),
    "emb2_norm_max": float(emb2_raw_norms.max()),
}

results_df = subset_df.copy()
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["residual"] = results_df["predicted_score_0_5"] - results_df["label"]
results_df["absolute_error"] = results_df["residual"].abs()
results_df["emb1_norm"] = emb1_raw_norms
results_df["emb2_norm"] = emb2_raw_norms

largest_errors_df = results_df.sort_values("absolute_error", ascending=False).reset_index(drop=True)

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5", "residual", "absolute_error"]].head(10))
print(embedding_norm_stats)
print(largest_errors_df[["sentence1", "sentence2", "label", "predicted_score_0_5", "residual", "absolute_error", "emb1_norm", "emb2_norm"]].head(10))


                              sentence1  \
0     A man with a hard hat is dancing.   
1      A young child is riding a horse.   
2  A man is feeding a mouse to a snake.   
3        A woman is playing the guitar.   
4         A woman is playing the flute.   
5          A woman is cutting an onion.   
6       A man is erasing a chalk board.   
7            A woman is carrying a boy.   
8        Three men are playing guitars.   
9               A woman peels a potato.   

                                  sentence2  label  cosine_similarity  \
0      A man wearing a hard hat is dancing.  5.000           0.996950   
1                A child is riding a horse.  4.750           0.973566   
2  The man is feeding a mouse to the snake.  5.000           0.945348   
3                  A man is playing guitar.  2.400           0.985312   
4                 A man is playing a flute.  2.750           0.983038   
5                  A man is cutting onions.  2.615           0.981624   
6       The man

In [7]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples_evaluated: {len(subset_df)}")
print("subset_rule: first_256_validation_examples_after_minimal_whitespace_cleaning_and_nonempty_filter")
print("embedding_rule: cls_token_last_hidden_state_then_l2_normalize_for_cosine")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"mean_absolute_error: {mae:.6f}")
print(f"avg_len1_words: {subset_df['len1_words'].mean():.2f}")
print(f"avg_len2_words: {subset_df['len2_words'].mean():.2f}")
print(f"avg_length_gap_words: {subset_df['len_gap_words'].mean():.2f}")
print(f"emb1_norm_mean: {embedding_norm_stats['emb1_norm_mean']:.6f}")
print(f"emb1_norm_std: {embedding_norm_stats['emb1_norm_std']:.6f}")
print(f"emb2_norm_mean: {embedding_norm_stats['emb2_norm_mean']:.6f}")
print(f"emb2_norm_std: {embedding_norm_stats['emb2_norm_std']:.6f}")
print(f"max_absolute_error: {results_df['absolute_error'].max():.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")

top_error_examples = largest_errors_df[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "residual", "absolute_error", "emb1_norm", "emb2_norm"
]].head(5)
print(top_error_examples.to_dict(orient="records"))


device_used: mps
model_name: distilbert-base-uncased
dataset_split: glue/stsb/validation
num_examples_evaluated: 256
subset_rule: first_256_validation_examples_after_minimal_whitespace_cleaning_and_nonempty_filter
embedding_rule: cls_token_last_hidden_state_then_l2_normalize_for_cosine
pearson_correlation: 0.238433
spearman_correlation: 0.314869
mean_absolute_error: 2.650924
avg_len1_words: 6.80
avg_len2_words: 6.71
avg_length_gap_words: 1.34
emb1_norm_mean: 12.463326
emb1_norm_std: 0.289591
emb2_norm_mean: 12.471003
emb2_norm_std: 0.280348
max_absolute_error: 4.937448
runtime_seconds: 2.76
[{'sentence1': 'A woman is riding on a horse.', 'sentence2': 'A man is shooting off guns.', 'label': 0.0, 'predicted_score_0_5': 4.937447547912598, 'residual': 4.937447547912598, 'absolute_error': 4.937447547912598, 'emb1_norm': 12.32437515258789, 'emb2_norm': 12.763912200927734}, {'sentence1': 'A man is cutting a potato.', 'sentence2': 'A woman is climbing a rock wall.', 'label': 0.0, 'predicted_sc